# Rakuten — Mistral-7B-v0.3 affiné en classifieur (QLoRA) sur GPU Colab

⚠ **Confidentialité** : les données du challenge sont confidentielles (mention légale de l'ENS / Rakuten).
Les déposer sur Colab revient à les envoyer chez Google. À toi de décider ; supprime l'environnement d'exécution à la fin.

**Avant de lancer** :
- Mistral est en accès restreint : accepte la licence sur https://huggingface.co/mistralai/Mistral-7B-v0.3,
  crée un token « Read » (huggingface.co → Settings → Access Tokens) et ajoute-le dans les **secrets Colab**
  (icône 🔑 à gauche, nom `HF_TOKEN`, accès au notebook activé).
- Exécution → Modifier le type d'exécution → **GPU L4** (ou A100).
- Sur Posos, Mistral seul était un peu sous Qwen (0,801 contre 0,812) mais **les deux ensemble** donnaient 0,829 :
  l'intérêt est la diversité. À comparer à Qwen sur le pli 0 de Rakuten : **0,9001**.

1. Cellule 1 (installation), puis cellule 2 : **dépose les 4 fichiers** :
   `X_train_update.csv`, `X_test_update.csv`, `Y_train_CVw08PX.csv`, `common.py` (du dossier `rakuten_multimodal/`).
2. Cellule 3 : choisis `FOLD` (0 à 4) et `MODELE`. Les plis sont exactement ceux de `common.py`.
3. Exécute tout. À la fin, télécharge `<NOM>_f<FOLD>.npz` et dépose-le dans `rakuten_multimodal/oof/`.
   Quand les 5 plis sont là : `python train_text.py --model x --name <NOM> --merge`, puis `stack.py`.

Durée indicative par pli (68 000 exemples d'entraînement, 1 epoch, `MAX_LEN` 128) :
A100 + Qwen2.5-7B ≈ 1 h 30 · T4 + Qwen2.5-1.5B ≈ 2 h · T4 + 7B ≈ 8 h+ (déconseillé).

In [ ]:
!pip -q install -U "transformers>=4.45,<4.47" "peft>=0.13" "bitsandbytes>=0.44" "accelerate>=0.34" datasets scikit-learn sentencepiece
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
from google.colab import files
import os
os.makedirs("data", exist_ok=True)
for nom in files.upload():
    if nom != "common.py":
        os.replace(nom, f"data/{nom}")
print(sorted(os.listdir("data")), os.path.exists("common.py"))

In [ ]:
FOLD = 0                     # 0..4 : un pli par session
MODELE = "mistralai/Mistral-7B-v0.3"   # ou "Qwen/Qwen2.5-7B"
NOM = "mistral7b"            # préfixe du fichier de sortie
EPOCHS = 1
LR = 1e-4
BATCH = 16
ACCUM = 1
MAX_LEN = 128
SEED = 42

# Modèles « gated » (Mistral, Llama…) : token Hugging Face rangé dans les secrets Colab, nom HF_TOKEN
from huggingface_hub import login
try:
    from google.colab import userdata
    login(userdata.get("HF_TOKEN"))
except Exception as e:
    print("Pas de token HF (inutile pour Qwen) :", type(e).__name__)

In [ ]:
import numpy as np, torch
from common import load, folds, wf1

tr, te, codes = load()                 # même nettoyage et mêmes étiquettes qu'en local
y = tr.label.values
a, b = folds(y)[FOLD]                   # mêmes plis qu'en local
print(len(a), len(b), len(te), len(codes), "classes")
bf16 = torch.cuda.is_bf16_supported()   # A100/L4 : bf16 ; T4 : fp16
dtype = torch.bfloat16 if bf16 else torch.float16

In [ ]:
from transformers import (AutoTokenizer, AutoModelForSequenceClassification, BitsAndBytesConfig,
                          Trainer, TrainingArguments, DataCollatorWithPadding, set_seed)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from datasets import Dataset
set_seed(SEED + FOLD)

tok = AutoTokenizer.from_pretrained(MODELE)
if tok.pad_token is None:
    tok.pad_token = tok.eos_token
bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                         bnb_4bit_compute_dtype=dtype, bnb_4bit_use_double_quant=True)
model = AutoModelForSequenceClassification.from_pretrained(
    MODELE, num_labels=len(codes), quantization_config=bnb, torch_dtype=dtype, device_map={"": 0})
model.config.pad_token_id = tok.pad_token_id
model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)
model = get_peft_model(model, LoraConfig(
    r=16, lora_alpha=32, lora_dropout=0.05, task_type="SEQ_CLS",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]))
model.print_trainable_parameters()

def ds(df, labels=None):
    d = {"text": list(df.text)}
    if labels is not None:
        d["label"] = labels
    return Dataset.from_dict(d).map(lambda x: tok(x["text"], truncation=True, max_length=MAX_LEN),
                                    batched=True, remove_columns=["text"])

ds_tr, ds_va, ds_te = ds(tr.iloc[a], y[a]), ds(tr.iloc[b], y[b]), ds(te)

In [ ]:
args = TrainingArguments(
    output_dir="out", num_train_epochs=EPOCHS, learning_rate=LR,
    per_device_train_batch_size=BATCH, per_device_eval_batch_size=32, gradient_accumulation_steps=ACCUM,
    warmup_ratio=0.03, lr_scheduler_type="cosine", weight_decay=0.01,
    bf16=bf16, fp16=not bf16, group_by_length=True, logging_steps=50, save_strategy="no",
    eval_strategy="no", report_to="none", seed=SEED + FOLD)
trainer = Trainer(model=model, args=args, train_dataset=ds_tr, data_collator=DataCollatorWithPadding(tok))
trainer.train()

In [ ]:
trainer.args.group_by_length = False   # sinon l'ordre des prédictions est permuté

def probs(d):
    return torch.from_numpy(trainer.predict(d).predictions).float().softmax(-1).numpy()

pv, pt = probs(ds_va), probs(ds_te)
print(f"{NOM} pli {FOLD} : F1 pondéré {wf1(y[b], pv.argmax(1)):.4f}   (Qwen 7B pli 0 : 0.9001 · CamemBERTav2 : 0.8954)")
nom = f"{NOM}_f{FOLD}.npz"
np.savez(nom, val=pv, test=pt)       # même format que train_text.py
files.download(nom)